# Solar Prediction Colab GRU Tuning

Tune the GRU on the full ignored dataset for the forecast horizons where recurrent models beat the simple baselines. This notebook intentionally skips LSTM and the 15-minute horizon.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Update this if your Drive folder name differs.
PROJECT_DIR = '/content/drive/MyDrive/solar_prediction_latest'
DATA_PATH = 'data/solar_weather.csv'
OUTPUT_ROOT = 'artifacts/colab_gru_tuning_actual_data'

BATCH_SIZE = 256
HORIZONS = {
    '4h': 16,
    '24h': 96,
}

SEARCH = [
    {'epochs': 50, 'hidden_dim': 32, 'batch_size': BATCH_SIZE},
    {'epochs': 100, 'hidden_dim': 32, 'batch_size': BATCH_SIZE},
    {'epochs': 100, 'hidden_dim': 64, 'batch_size': BATCH_SIZE},
    {'epochs': 150, 'hidden_dim': 64, 'batch_size': BATCH_SIZE},
    {'epochs': 100, 'hidden_dim': 128, 'batch_size': BATCH_SIZE},
]


In [ ]:
import os
from pathlib import Path

project = Path(PROJECT_DIR)
assert project.exists(), f'Project folder not found: {project}'
os.chdir(project)

data_path = Path(DATA_PATH)
assert data_path.exists(), f'Full dataset not found: {data_path}'

print('Working directory:', Path.cwd())
print('Full dataset:', data_path)


Use a fresh Colab runtime. This notebook installs the local package only and leaves Colab's scientific Python stack alone.

In [ ]:
%pip install -q --no-deps -e .


In [ ]:
import pandas as pd

preview = pd.read_csv(data_path, nrows=5)
time_col = pd.read_csv(data_path, usecols=['Time'])['Time']
parsed_time = pd.to_datetime(time_col, errors='coerce')

print('Rows:', len(time_col))
print('Columns:', preview.columns.tolist())
print('Time range:', parsed_time.min(), 'to', parsed_time.max())
preview.head()


In [ ]:
from io import StringIO
import shlex
import subprocess
import sys

import pandas as pd
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
all_metrics = []
output_root = Path(OUTPUT_ROOT)
output_root.mkdir(parents=True, exist_ok=True)

for horizon_label, horizon_steps in HORIZONS.items():
    for cfg in SEARCH:
        run_name = f"e{cfg['epochs']:03d}_h{cfg['hidden_dim']:03d}_b{cfg['batch_size']}"
        output_dir = output_root / horizon_label / run_name
        cmd = [
            sys.executable,
            '-m',
            'solar_prediction.cli',
            'train',
            '--model', 'gru',
            '--data', str(data_path),
            '--epochs', str(cfg['epochs']),
            '--hidden-dim', str(cfg['hidden_dim']),
            '--batch-size', str(cfg['batch_size']),
            '--horizon-steps', str(horizon_steps),
            '--device', device,
            '--output', str(output_dir),
            '--quiet',
        ]

        print('\nHorizon:', horizon_label, f'({horizon_steps} steps)', 'Run:', run_name)
        print('Command:', shlex.join(cmd))
        result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        print(result.stdout)
        output_dir.mkdir(parents=True, exist_ok=True)
        (output_dir / 'command_output.txt').write_text(result.stdout)
        if result.returncode != 0:
            raise RuntimeError(f'Command failed with exit code {result.returncode}')

        checkpoint = output_dir / 'gru_model.pt'
        metadata = output_dir / 'gru_metadata.json'
        assert checkpoint.exists(), f'Missing checkpoint: {checkpoint}'
        assert metadata.exists(), f'Missing metadata: {metadata}'

        output_lines = result.stdout.splitlines()
        header_idx = next(i for i, line in enumerate(output_lines) if line.startswith('model,'))
        metrics = pd.read_csv(StringIO('\n'.join(output_lines[header_idx:])))
        metrics.insert(0, 'horizon', horizon_label)
        metrics.insert(1, 'horizon_steps', horizon_steps)
        metrics.insert(2, 'run_name', run_name)
        metrics.insert(3, 'epochs', cfg['epochs'])
        metrics.insert(4, 'hidden_dim', cfg['hidden_dim'])
        metrics.insert(5, 'batch_size', cfg['batch_size'])
        metrics['output_dir'] = str(output_dir)
        metrics['checkpoint'] = str(checkpoint)
        all_metrics.append(metrics)

tuning_metrics = pd.concat(all_metrics, ignore_index=True)
tuning_metrics.to_csv(output_root / 'gru_tuning_metrics.csv', index=False)
tuning_metrics.sort_values(['horizon_steps', 'rmse']).reset_index(drop=True)


In [ ]:
best_by_horizon = (
    tuning_metrics.sort_values('rmse')
    .groupby('horizon', sort=False)
    .first()
    .reset_index()
)
best_by_horizon.to_csv(output_root / 'gru_tuning_best_by_horizon.csv', index=False)
best_by_horizon[
    ['horizon', 'horizon_steps', 'run_name', 'epochs', 'hidden_dim', 'batch_size', 'rmse', 'mae', 'r2', 'capped_mape', 'checkpoint']
]


In [ ]:
import matplotlib.pyplot as plt

plot_df = tuning_metrics.copy()
plot_df['config'] = plot_df['run_name']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for horizon_label in HORIZONS:
    subset = plot_df[plot_df['horizon'] == horizon_label]
    axes[0].plot(subset['config'], subset['rmse'], marker='o', label=horizon_label)
    axes[1].plot(subset['config'], subset['r2'], marker='o', label=horizon_label)

axes[0].set_title('GRU tuning RMSE')
axes[0].set_xlabel('')
axes[0].set_ylabel('W/m^2')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend()

axes[1].set_title('GRU tuning R2')
axes[1].set_xlabel('')
axes[1].set_ylabel('')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()

plt.tight_layout()
plt.show()
